# Multimodal Product Price Predictor (Amazon ML Challenge 2025)

Predicts a product's price from its **catalog text** and **product image**.

- Text encoder: `bert-base-uncased` (pooled output)
- Image encoder: `google/vit-base-patch16-224` (pooled output)
- Both projected to 512-d, concatenated, then an MLP regression head
- Loss: SMAPE (the challenge metric), optimiser AdamW, cosine-annealing LR

This approach placed **AIR 38 among 9,000+ teams**. This notebook is a cleaned, reproducible version of the competition code.

Run `download_images.py` first so images are read from disk instead of over HTTP during training.

In [ ]:
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel, ViTModel
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR  = "dataset"          # contains train.csv / test.csv from the challenge
IMAGE_DIR = "images"           # filled by download_images.py ({sample_id}.jpg)
TEXT_MODEL  = "bert-base-uncased"
IMAGE_MODEL = "google/vit-base-patch16-224"
MAX_LEN, BATCH_SIZE, EPOCHS, LR = 128, 16, 5, 2e-5

## Data
Prices are heavily right-skewed, so the network predicts `log(price)` and predictions are mapped back with `exp`.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# hold out 10% of train for validation (test.csv has no prices)
val_df   = train_df.sample(frac=0.1, random_state=SEED)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
img_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ProductPricingDataset(Dataset):
    def __init__(self, df, train=True):
        self.df, self.train = df, train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        enc = tokenizer(str(row["catalog_content"]), truncation=True, padding="max_length",
                        max_length=MAX_LEN, return_tensors="pt")
        path = os.path.join(IMAGE_DIR, f"{row['sample_id']}.jpg")
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), (255, 255, 255))   # missing image -> blank
        item = {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "image": img_tf(img),
                "sample_id": row["sample_id"]}
        if self.train:
            item["target"] = torch.tensor(np.log(max(float(row["price"]), 1e-2)), dtype=torch.float)
        return item

workers = min(4, os.cpu_count() or 1)
train_loader = DataLoader(ProductPricingDataset(train_df), BATCH_SIZE, shuffle=True,  num_workers=workers)
val_loader   = DataLoader(ProductPricingDataset(val_df),   BATCH_SIZE, shuffle=False, num_workers=workers)
test_loader  = DataLoader(ProductPricingDataset(test_df, train=False), BATCH_SIZE, shuffle=False, num_workers=workers)

## Model: BERT + ViT fusion

In [ ]:
class MultiModalRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_model  = AutoModel.from_pretrained(TEXT_MODEL)
        self.image_model = ViTModel.from_pretrained(IMAGE_MODEL)
        self.text_proj   = nn.Linear(self.text_model.config.hidden_size, 512)
        self.image_proj  = nn.Linear(self.image_model.config.hidden_size, 512)
        self.head = nn.Sequential(nn.Linear(1024, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))

    def forward(self, input_ids, attention_mask, images):
        t = self.text_proj(self.text_model(input_ids=input_ids, attention_mask=attention_mask).pooler_output)
        v = self.image_proj(self.image_model(pixel_values=images).pooler_output)
        return self.head(torch.cat([t, v], dim=1)).squeeze(-1)

## SMAPE
SMAPE is the competition metric. Training minimises SMAPE on the price scale (the model predicts log-price; `exp` keeps every prediction positive).

In [ ]:
def smape(pred, true, eps=1e-6):
    return (torch.abs(pred - true) / ((torch.abs(true) + torch.abs(pred)) / 2 + eps)).mean()

def to_price(log_pred):
    # exp keeps prices positive and gives a gradient everywhere (unlike clamping at 0)
    return torch.exp(log_pred.clamp(max=15))

## Train

In [ ]:
model = MultiModalRegressor().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train):
    model.train(train)
    total, n = 0.0, 0
    for b in tqdm(loader, leave=False):
        ids, mask, img = b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), b["image"].to(DEVICE)
        y = b["target"].to(DEVICE)
        with torch.set_grad_enabled(train):
            pred = model(ids, mask, img)
            loss = smape(to_price(pred), torch.exp(y))
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(y); n += len(y)
    return 100 * total / n

best = float("inf")
for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    scheduler.step()
    print(f"epoch {epoch+1}/{EPOCHS}  train SMAPE {tr:.2f}%  val SMAPE {va:.2f}%")
    if va < best:
        best = va
        torch.save(model.state_dict(), "best_model.pth")

## Predict and write the submission

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()
ids_out, prices = [], []
with torch.no_grad():
    for b in tqdm(test_loader):
        pred = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), b["image"].to(DEVICE))
        prices += to_price(pred).cpu().tolist()
        ids_out += list(b["sample_id"])
pd.DataFrame({"sample_id": ids_out, "price": prices}).to_csv("test_out.csv", index=False)
print("saved test_out.csv")